# SFR Convex Linear Structure Screener

Ranks 3M SOFR futures calendar spreads and butterflies by the asymmetry of
their option-implied payoff distribution. See
`docs/plans/2026-04-28-sfr-convex-screener-design.md` for the design.

**Caveat:** Risk-neutral ≠ real-world. Reported asymmetry ratios reflect the
option-implied measure; real-world distributions differ by the price of risk.
Joint dependence is the largest single source of model error — review both
`common_state` and `historical_gaussian_copula` columns when evaluating any
structure.

In [ ]:
import datetime
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytz

from RVUtils.SFRConvexScreener import (
    JointMethod,
    SFRConvexScreenerConfig,
    build_snapshot,
    write_snapshot,
)
from RVUtils.SFRConvexScreener._distributions import (
    payoff_pdf_common_state,
    payoff_pdf_historical_gaussian_copula,
    payoff_pdf_perfect_correlation,
)

NYC = pytz.timezone('America/New_York')
as_of = datetime.date.today()
print(f'as_of = {as_of}')

## Configuration

In [ ]:
config = SFRConvexScreenerConfig(
    universe_size=12,
    calendar_gaps=(1, 2, 4),
    fly_gaps=(1, 2, 4),
    primary_joint_method=JointMethod.COMMON_STATE,
)
config

## Build snapshot

Loads market data (curve, futures snapshot, options smiles, 60d price
panel), enumerates the universe, computes payoff PDFs under three joint
assumptions, scores cross-sectionally, and persists the snapshot.

In [ ]:
snap = build_snapshot(config, as_of=as_of)
print(f'{len(snap.results)} structures ranked')
for w in snap.run_warnings[:10]:
    print(' run-warning:', w)

## Top 20 by composite score

In [ ]:
df = snap.to_dataframe().sort_values('composite_score', ascending=False)
df.head(20).style.format(precision=2, na_rep='-')

## Top 10 by asymmetry alone

In [ ]:
df.sort_values('asymmetry_ratio', ascending=False).head(10).style.format(precision=2, na_rep='-')

## Top 5 payoff PDFs (primary method)

In [ ]:
top5 = list(snap.results)[:5]
fig, axes = plt.subplots(len(top5), 1, figsize=(10, 3 * len(top5)), sharex=False)
if len(top5) == 1:
    axes = [axes]
for ax, r in zip(axes, top5):
    metrics = r.metrics_by_method[r.primary_method]
    ax.set_title(
        f"{r.structure_def.structure_id}  rank={r.rank}  A={metrics.asymmetry_ratio:.2f}  E[P&L]={metrics.mean_bp:.2f} bp"
    )
    pcts = metrics.percentiles_bp
    ax.axvspan(pcts['p5'], pcts['p95'], alpha=0.15, label='5–95%')
    ax.axvspan(pcts['p25'], pcts['p75'], alpha=0.30, label='25–75%')
    ax.axvline(metrics.mean_bp, color='red', linestyle='--', label='mean')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('P&L (bp)')
    ax.set_ylabel('density (sketch)')
    ax.legend(loc='upper right', fontsize=8)
fig.tight_layout()
plt.show()

## IV/RV diagnostics across the strip

In [ ]:
ivrv_rows = []
for r in snap.results:
    for d in r.iv_rv_diagnostics:
        ivrv_rows.append({
            'structure_id': r.structure_def.structure_id,
            'contract': d.contract,
            'iv_bp': d.iv_bp,
            'rv_bp': d.rv_bp,
            'iv_rv_ratio': d.iv_rv_ratio,
        })
ivrv_df = pd.DataFrame(ivrv_rows)
if not ivrv_df.empty:
    by_contract = ivrv_df.groupby('contract')[['iv_bp', 'rv_bp', 'iv_rv_ratio']].mean()
    by_contract.style.format(precision=2)
else:
    print('no iv/rv diagnostics')

## Persist snapshot to disk

In [ ]:
paths = write_snapshot(snap, root_dir=Path(config.output_root))
for k, p in paths.items():
    print(f'{k:>4}: {p}')

## Caveats

* Risk-neutral ≠ real-world. Asymmetry ratios shown are RN-implied.
* Joint dependence model risk: review the `common_state`,
  `historical_gaussian_copula`, and `perfect_correlation` columns of each
  result for sensitivity to the joint assumption.
* BL-extracted tails depend on SABR + ghost-point extrapolation. Per-leg
  warnings (clipping, rate-floor truncation) are surfaced in
  `result.warnings`.
* Carry / roll-down are placeholder (NaN) until the IRSwapQuery path is
  wired (Phase 5).